In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

In [46]:

import os
import pandas as pd
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from src.RAG.Constants.models import GroqModelList

gml = GroqModelList()

load_dotenv('../../Secrets/RAG.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY','')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','')
print(len(GROQ_API_KEY))

56


In [10]:
llm_groq = ChatGroq(
    model=gml.openai.gpt_oss_20b,
    temperature=0.7,
    api_key=GROQ_API_KEY,    
)
llm_groq.invoke(input='What LLM model are you?')

AIMessage(content='I’m ChatGPT, built on OpenAI’s GPT‑4 architecture.', additional_kwargs={'reasoning_content': 'The user asks: "What LLM model are you?" The system instructions say: "You are ChatGPT, a large language model trained by OpenAI." So the answer should be: I\'m ChatGPT, based on GPT-4 architecture. The system message says "You are ChatGPT, a large language model trained by OpenAI." So answer accordingly. Also we need to follow the policy. No disallowed content. It\'s safe. So respond: I\'m ChatGPT, GPT-4.'}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 78, 'total_tokens': 202, 'completion_time': 0.150761309, 'completion_tokens_details': {'reasoning_tokens': 100}, 'prompt_time': 0.005121288, 'prompt_tokens_details': None, 'queue_time': 0.052958442, 'total_time': 0.155882597}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'gro

In [11]:
graph_schema = """
node_label {node_property: dtype}:
Country {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
State {ids: str, name: str, iso_code: str, coords: list[float], boundingbox: list[float]}
City {ids: str, name: str, coords: list[float], boundingbox: list[float]}
Area {ids: str, name: str}
Locality {ids: str, name: str}
Restaurant {ids: int, name: str, city: str, area: str, locality: str, cuisines: List[str], rating: float | null, address: str, coords: List[float] | null, chain: bool, city_id: str}
Menu {name: str, category: str, description: str, types: Literal["VEG", "NONVEG", "EGG", "UNKNOWN"], cuisine: str}
MainCuisine {name: str}

link_label {link_property: dtype}:
HAS_STATE {}
HAS_CITY {}
HAS_AREA {}
HAS_LOCALITY {}
HAS_RESTAURANT {}
HAS_MENU {price: int | null, rating: float | null}
SERVES_MAIN_CUISINE {}

Relationships:
(:Country)-[:HAS_STATE]->(:State)
(:State)-[:HAS_CITY]->(:City)
(:City)-[:HAS_AREA]->(:Area)
(:Area)-[:HAS_LOCALITY]->(:Locality)
(:Locality)-[:HAS_RESTAURANT]->(:Restaurant)
(:Restaurant)-[:HAS_MENU]->(:Menu)
(:Restaurant)-[:SERVES_MAIN_CUISINE]->(:MainCuisine)
"""

In [ ]:
cypher_pattern = """
Allowed query patterns:

# Search for Restaurants based on location change to city.ids
PATTERN 1.1: Search for Restaurants by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.2: Search for Restaurants by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

PATTERN 1.3: Search for Restaurants by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->(r:Restaurant)
RETURN r LIMIT 5000 DESC r.ids

# Search for Menus based on location
PATTERN 2.1: Search for Menus by City
MATCH (:City {name:$city})-[]-()-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m LIMIT 5000 DESC m.name

PATTERN 2.2: Search for Menus by Area
MATCH (:Area {name:$area})-[]-()-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

PATTERN 2.3: Search for Menus by Locality
MATCH (:Locality {name:$locality})-[:HAS_RESTAURANT]->()-[:HAS_MENU]-(m:Menu)
RETURN m

# Search for Menus based on cuisines



PATTERN_2: Restaurant by cuisine
MATCH (r:Restaurant)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_3: Restaurant by city AND cuisine
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City {name:$city}),
      (r)-[:SERVES]->(cu:Cuisine {name:$cuisine})
RETURN r

PATTERN_4: Similar restaurants
MATCH (r:Restaurant {name:$name})-[s:SIMILAR_TO]->(other)
RETURN other ORDER BY s.score DESC

Task:
- Select the SINGLE best pattern
- Output ONLY:
  pattern_name
  parameter_values

"""



In [ ]:

system_prompt = f"""
You are a graph query planner.

You MUST follow these rules:
- Use ONLY the node labels listed below.
- Use ONLY the relationship types listed below.
- Use ONLY the properties listed below.
- Do NOT invent new labels, relationships, or properties.
- If a question cannot be answered using this schema, respond with:
  "NOT_ANSWERABLE_WITH_SCHEMA"

Graph schema:
{graph_schema}

You are NOT allowed to use any other schema elements.

"""

In [12]:
from src.ETL.Config.graph_pool import GraphPool

graph = GraphPool.get_graph(graph_name='test')
graph

In [84]:
graph_query = """
MATCH (:City {ids: 'relation:7902476'})-[*3]->(obj:Restaurant)
WHERE obj.rating > 4.5
RETURN DISTINCT properties(obj) AS obj_props
ORDER BY obj_props.ids ASC
LIMIT 200
"""

result = graph.query(graph_query).result_set
df = pd.DataFrame(data=[row[0] for row in result], columns=result[0][0].keys())
df

,address,area,chain,city,city_id,coords,cuisines,ids,locality,name,rating
0,"143/A, 60 Feet Road, 5Th Block, Koramangala, B...",Koramangala,True,Bangalore,bangalore,"[12.93716728, 77.61962126]","[Sweets, Snacks]",218,Old Airport Road,Anand Sweets & Savouries,4.6
1,"93/A, Appek Building, 'A' Wing, 4Th 'B' Cross...",Koramangala,True,Bangalore,bangalore,"[12.93350024, 77.61430476]","[American, Continental]",223,Koramangala,Truffles,4.6
2,"124, 1st A Cross Rd, Near Jyoti Nivas College ...",Koramangala,True,Bangalore,bangalore,"[12.93449804, 77.6161871]","[Biryani, Andhra]",229,Koramangala,Meghana Foods,4.6
3,"4, 8Th Main Road, 4Th Block, Koramangala 4Th B...",Koramangala,False,Bangalore,bangalore,"[12.9345782537974, 77.6255145433083]",[American],232,Koramangala,The Hole In The Wall Cafe,4.6
4,"Kim Lee Restaurant, Above Sufyan, #50, 1st Flo...",Ulsoor,False,Bangalore,bangalore,"[12.97030241, 77.63259485]","[Chinese, Seafood]",246,Indiranagar,Kim Lee,4.6
...,...,...,...,...,...,...,...,...,...,...,...
195,"#2984, Near SBI, 12TH Main Road, Hal 2nd Stage...",Indiranagar,True,Bangalore,bangalore,"[12.970216, 77.644605]","[Ice Cream, Desserts]",29668,Indiranagar,Natural Ice Cream,4.8
196,"65/1 C, Shivaganga Complex, BESIDE Just Books,...",Sarjapur Road,True,Bangalore,bangalore,"[12.9129109, 77.68027924]","[Ice Cream, Desserts]",29670,Sarjapur Road,Natural Ice Cream,4.8
197,"#31, Pandurangnagar, 60 Feet Road, Adjacent to...",Arekere,True,Bangalore,bangalore,"[12.8921763, 77.5981881]","[Ice Cream, Desserts]",29671,Pandurangnagar,Natural Ice Cream,4.8
198,"#9/10, Das Commercial Complex, NEAR JSS Circle...",Jayanagar,True,Bangalore,bangalore,"[12.92394946, 77.57651479]","[Ice Cream, Desserts]",29672,Jayanagar,Natural Ice Cream,4.8


In [114]:
# get best rated menus from best rated restaurants serving thai cuisines in Koramangala
from time import time


graph_query = """
MATCH (:Area {name: 'Koramangala'})-[*2]->(rstn:Restaurant)-[]->(:MainCuisine {name: 'Thai'})
WHERE rstn.rating > 4.0
MATCH (rstn)-[link:HAS_MENU]-(menu:Menu)
WHERE link.rating > 4.0
RETURN DISTINCT
    rstn.name,
    menu.name,
    menu.types,
    link.price,
    link.rating
ORDER BY link.rating DESC
LIMIT 2000
"""
st = time()
result = graph.query(graph_query).result_set
print(f"Time taken for graph query: {(time() - st)*1000:.2f} ms")
df = pd.DataFrame([row for row in result], columns=['Restautant','Name', 'Type', 'Price', 'Rating'])


df.head(10)

Time taken for graph query: 247.40 ms


,Restautant,Name,Type,Price,Rating
0,Bamey's Restro Cafe,Paneer Chilly,VEG,315.0,5.0
1,Momo Rice Noodles,Fish Manchurian Gravy,NONVEG,370.0,5.0
2,Momo Rice Noodles,Veg Chilli Garlic Curry with Rice,VEG,250.0,5.0
3,Momo Rice Noodles,Chilli Crispy Fish Dry,NONVEG,370.0,5.0
4,Momo Rice Noodles,Spicy Curd Chicken Lollipop ( 6 pic),NONVEG,331.0,5.0
5,Momo Rice Noodles,Veg Butter Garlic Noodles,VEG,225.0,5.0
6,Beijing Bites,Hunan Tofu,VEG,299.0,5.0
7,Momo Rice Noodles,Pork Schezwan Fried Rice,NONVEG,290.0,5.0
8,Momo Rice Noodles,Seafood Mushroom Clear Soup,NONVEG,195.0,5.0
9,Thai Basil,Stir Fried Chicken Red Curry Paste,NONVEG,480.0,5.0


In [102]:
df.shape

(683, 5)

In [205]:
from typing import Literal, Dict, Any, List


In [208]:
def get_competitors(
    q_params: dict,
    output: Literal['dict', 'dataframe'] = 'dict',
    exclude: List[str]=["ids", "city_id"],
) -> List[Dict[str, Any]] | pd.DataFrame:
    graph_query = """
    MATCH (:Area {name: $area})-[:HAS_LOCALITY]->(:Locality)-[:HAS_RESTAURANT]->(r:Restaurant)
    MATCH (r)-[:SERVES_MAIN_CUISINE]->(:MainCuisine {name: $cuisine})
    WHERE r.rating IS NOT NULL AND r.rating >= $min_rating
    RETURN DISTINCT r
    ORDER BY r.rating DESC, r.name ASC
    LIMIT $limit
    """
    result = graph.query(graph_query, q_params).result_set
    rqrd_data = [
        {
            key: ', '.join([str(item) for item in val]) if isinstance(val, list) else val
            for key, val in res[0].properties.items() 
            if key not in exclude
        }
        for res in result
    ]
    # include query time in logs
    return pd.DataFrame(rqrd_data) if output=='dataframe' else rqrd_data

In [209]:
from src.RAG.Config.tool_models import GetCompetitorModels

f_params = GetCompetitorModels.FunctionParams(
    q_params=GetCompetitorModels.QueryParams(
        area='Indiranagar',
        cuisine='Thai',
        limit=200,
    ),
    output='dataframe',
    exclude=['ids','city_id'],
)
cmpt_data = get_competitors(**f_params.model_dump())
cmpt_data

,name,city,area,locality,cuisines,rating,address,coords,chain
0,Thai Basil,Bangalore,Indiranagar,Indiranagar,Thai,4.6,840/A/1 Indiranagar 100Ft Road 560038,"12.9806363, 77.6407933",True
1,SHARON TEA STALL,Bangalore,Indiranagar,Indiranagar,"Thai, Bakery",4.5,"Shop No : 1 , Floor : G , SHARON TEA STALL , N...","12.9731234486116, 77.6471368762574",True
2,Xian,Bangalore,Indiranagar,Sarvagna Nagar,"Chinese, Thai",4.5,"# 11,Haudin Road, Diagonally opp God's Gift Ap...","12.961136, 77.635647",False
3,Beijing Bites,Bangalore,Indiranagar,CMH Road,"Chinese, Thai",4.4,"537/2, CMH Road, Indiranagar, Bangalore","12.977945, 77.638814",True
4,Tiger Thai,Bangalore,Indiranagar,100 feet road,"Thai, Pan-Asian",4.4,"No.656, TAIKI, 100 FEET ROAD , INDIRANGAR , BA...","12.9769721, 77.6406962",False
5,Wok In Chow - Pure Vegetarian Asian Street Food,Bangalore,Indiranagar,Appareddipalya,"Chinese, Thai",4.4,"XJCP+4CG, 11th Main Rd, Appareddipalya, Indira...","12.970307402356, 77.636133347885",False
6,China Garten,Bangalore,Indiranagar,Hal 2dd Stage,"Chinese, Thai",4.3,No 82 ground floor vasanthappa garden 13th g m...,"12.965888, 77.639993",False
